In [17]:
import cv2
import numpy as np
import time
from math import atan2, degrees

video_path = "people.mp4"

trackers = [('CSRT', cv2.TrackerCSRT_create), 
            ('KCF', cv2.TrackerKCF_create), 
            ('MOSSE', cv2.legacy.TrackerMOSSE_create)]

all_results = []

for tracker_name, tracker_func in trackers:
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    
    print(f"\n=== {tracker_name} ===")
    bbox = cv2.selectROI(f"{tracker_name} - выберите объект", frame, False)
    cv2.destroyAllWindows()
    
    if bbox != (0,0,0,0):
        tracker = tracker_func()
        tracker.init(frame, bbox)
        
        traj = []
        x,y,w,h = bbox
        prev = (x+w//2, y+h//2)
        
        frames = 0
        lost_frames = 0
        speeds = []
        start = time.time()
        
        while True:
            ret, frame = cap.read()
            if not ret: break
            
            ok, b = tracker.update(frame)
            
            if ok:
                x,y,w,h = [int(v) for v in b]
                curr = (x+w//2, y+h//2)
                traj.append(curr)
                if len(traj) > 30: traj.pop(0)
                
                dx, dy = curr[0]-prev[0], curr[1]-prev[1]
                sp = (dx*dx+dy*dy)**0.5
                speeds.append(sp)
                ang = degrees(atan2(dy,dx)) if sp>0 else 0
                prev = curr
                
                cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
                cv2.circle(frame, curr, 4, (0,0,255), -1)
                if len(traj) > 1:
                    cv2.polylines(frame, [np.array(traj, dtype=np.int32)], False, (0,255,255), 2)
                cv2.putText(frame, f"{tracker_name} | Speed:{sp:.1f} | Angle:{ang:.0f}", (10,30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            else:
                lost_frames += 1
                cv2.putText(frame, "LOST", (300,240), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
            
            cv2.imshow(tracker_name, cv2.resize(frame, (640,480)))
            frames += 1
            if cv2.waitKey(1) & 0xFF == 27: break
        
        cap.release()
        cv2.destroyAllWindows()
        
        fps = frames / (time.time()-start)
        all_results.append([tracker_name, round(fps,1), round(sum(speeds)/len(speeds),1) if speeds else 0, lost_frames])
        print(f"{tracker_name}: FPS={fps:.1f}, Потеряно={lost_frames}")

print("\n=== РЕЗУЛЬТАТЫ ===")
print("Трекер   | FPS | Скорость | Потеряно")
print("-" * 40)
for r in all_results:
    print(f"{r[0]:8} | {r[1]:4} | {r[2]:7} | {r[3]}")


=== CSRT ===
CSRT: FPS=19.5, Потеряно=0

=== KCF ===
KCF: FPS=32.5, Потеряно=0

=== MOSSE ===
MOSSE: FPS=63.5, Потеряно=1

=== РЕЗУЛЬТАТЫ ===
Трекер   | FPS | Скорость | Потеряно
----------------------------------------
CSRT     | 19.5 |     1.8 | 0
KCF      | 32.5 |     0.6 | 0
MOSSE    | 63.5 |     0.2 | 1
